In [ ]:
# ============================================================
# BƯỚC 1: CÀI ĐẶT THƯ VIỆN & CẤU HÌNH VRAM GPU T4
# ============================================================
import os
# 1. Cấu hình chống phân mảnh VRAM cho GPU T4 (Tránh lỗi CUDA OutOfMemory)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# 2. Gỡ bỏ gói torchaudio thừa để triệt tiêu hoàn toàn xung đột CUDA
!pip uninstall -y torchaudio -q

# 3. Cài đặt các thư viện cần thiết cho Qwen 3-VL Vision-Language Model
!pip install -q --upgrade transformers accelerate qwen-vl-utils

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("⚠️ CẢNH BÁO: Chưa bật GPU! Hãy vào menu: Thời lượng chạy (Runtime) -> Thay đổi loại thời lượng chạy -> Chọn T4 GPU.")


In [ ]:
# ============================================================
# BƯỚC 2: NẠP MÔ HÌNH QWEN 3-VL 4B INSTRUCT (CHUYÊN DỤNG VIDEO & THỊ GIÁC)
# ============================================================
import sys
import torch

# Phòng thủ kép: Tự động vô hiệu hóa torchaudio nếu phát hiện xung đột CUDA version
try:
    import torchaudio
except Exception:
    try:
        import transformers.utils.import_utils as _iu
        _iu._torchaudio_available = False
    except Exception:
        pass

try:
    from transformers import Qwen3VLForConditionalGeneration as ModelClass
except ImportError:
    try:
        from transformers import AutoModelForImageTextToText as ModelClass
    except ImportError:
        try:
            from transformers import AutoModelForVision2Seq as ModelClass
        except ImportError:
            from transformers import AutoModelForMultimodalLM as ModelClass

from transformers import AutoProcessor

# Sử dụng chính thức mô hình Qwen3-VL-4B-Instruct chuyên biệt cho Video-LLM từ Alibaba
model_id = "Qwen/Qwen3-VL-4B-Instruct"
print(f"⏳ Đang nạp mô hình {model_id} (Trọng số ~8 GB)...\n")

model = ModelClass.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)

# Thiết lập giới hạn pixel tối ưu cho chuỗi 8 frame trên GPU T4 (Tiết kiệm 2 GB VRAM)
min_pixels = 256 * 28 * 28
max_pixels = 512 * 28 * 28
processor = AutoProcessor.from_pretrained(
    model_id,
    min_pixels=min_pixels,
    max_pixels=max_pixels,
    trust_remote_code=True
)

print(f"✅ Mô hình {model_id} đã nạp thành công vào GPU T4 với kiến trúc Qwen3-VL chuyên dụng!")


In [ ]:
# ============================================================
# BƯỚC 3: SUY LUẬN VLM TỰ ĐỘNG (CHUẨN HỌC THUẬT VIDVRD + GIẢI THÍCH REASON)
# ============================================================
import os
import json
import glob
import torch
from PIL import Image
from qwen_vl_utils import process_vision_info

# 1. Tự động đọc Prompt & Danh sách Frames từ file Payload JSON (Ưu tiên nạp chuẩn xác)
payload_files = sorted(
    [p for p in glob.glob('/content/*payload*.json') + glob.glob('/content/*.json') if 'ket_qua' not in p and 'sample_data' not in p],
    key=os.path.getmtime,
    reverse=True
)

target_frame_names = []
if payload_files:
    payload_path = payload_files[0]
    print(f'📄 Tự động tải cấu hình từ payload: {payload_path}')
    with open(payload_path, 'r', encoding='utf-8') as f:
        payload_data = json.load(f)
    system_prompt = payload_data.get('vlm_system_prompt', '')
    user_prompt = payload_data.get('vlm_user_prompt', '')
    allowed_relations = payload_data.get('allowed_relations_vocabulary_26', [])
    target_frame_names = payload_data.get('visual_prompt_frames_sequence', [])
else:
    print('ℹ️ Không tìm thấy payload, sử dụng prompt mặc định tổng quát.')
    allowed_relations = ['bite', 'carry', 'clean', 'cut', 'drive', 'feed', 'get_off', 'get_on', 'grab', 'hit', 'hold', 'hug', 'kick', 'kiss', 'knock', 'lean_on', 'lick', 'lift', 'play(instrument)', 'pull', 'push', 'ride', 'shake_hand_with', 'throw', 'touch', 'wave']
    allowed_objects_60 = ['bird', 'cattle', 'dog', 'horse', 'lizard', 'rabbit', 'sheep', 'snake', 'turtle', 'chicken', 'duck', 'cat', 'pig', 'goat', 'person', 'child', 'bicycle', 'bus', 'car', 'motorcycle', 'train', 'baby_seat', 'baby_walker', 'stop_sign', 'traffic_light', 'truck', 'scooter', 'ball', 'skateboard', 'sofa', 'bread', 'cake', 'dish', 'fruits', 'vegetables', 'backpack', 'camera', 'cellphone', 'handbag', 'laptop', 'suitcase', 'bat', 'racket', 'toy', 'bottle', 'chair', 'cup', 'electric_fan', 'faucet', 'sink', 'oven', 'microwave', 'refrigerator', 'screen', 'stool', 'table', 'toilet', 'guitar', 'piano', 'bench']
    system_prompt = (
        "You are an advanced Video Visual Relation Detection (VidVRD) AI for surveillance analytics. You are given a temporal sequence of video frames with numbered visual marks [ID] identifying subjects and objects. Your task is to detect all active visual relations occurring between the marked entities over time.\n"
        "\n"
        "STRICT CONSTRAINTS:\n"
        "1. You MUST strictly select relation predicates ONLY from these 26 predefined categories: ['bite', 'carry', 'clean', 'cut', 'drive', 'feed', 'get_off', 'get_on', 'grab', 'hit', 'hold', 'hug', 'kick', 'kiss', 'knock', 'lean_on', 'lick', 'lift', 'play(instrument)', 'pull', 'push', 'ride', 'shake_hand_with', 'throw', 'touch', 'wave']. All other verbs are strictly prohibited.\n"
        "2. Entity subject and object classes belong strictly to the 60 predefined categories: ['bird', 'cattle', 'dog', 'horse', 'lizard', 'rabbit', 'sheep', 'snake', 'turtle', 'chicken', 'duck', 'cat', 'pig', 'goat', 'person', 'child', 'bicycle', 'bus', 'car', 'motorcycle', 'train', 'baby_seat', 'baby_walker', 'stop_sign', 'traffic_light', 'truck', 'scooter', 'ball', 'skateboard', 'sofa', 'bread', 'cake', 'dish', 'fruits', 'vegetables', 'backpack', 'camera', 'cellphone', 'handbag', 'laptop', 'suitcase', 'bat', 'racket', 'toy', 'bottle', 'chair', 'cup', 'electric_fan', 'faucet', 'sink', 'oven', 'microwave', 'refrigerator', 'screen', 'stool', 'table', 'toilet', 'guitar', 'piano', 'bench'].\n"
        "3. SYSTEMATIC INTERACTION RULES:\n"
        "   - Person-Person Contact: Active physical contact between persons (such as hands touching shoulders, arms, or bodies) is categorized as 'touch'.\n"
        "   - Person-Object Manipulation: When a person holds and transports an object while moving or walking across frames, categorize as 'carry'. When a person holds an object statically in hand(s), categorize as 'hold'.\n"
        "   - Zero-Displacement Inactive Clutter: If an object remains completely stationary in the exact same location across ALL frames without any movement or displacement, OMIT that pair entirely (a person merely walking past or standing near a stationary object on the floor/surface is NOT an interaction).\n"
        "   - Temporal Transitions: If a person actively carries or holds an object in ANY frames, report that valid interaction even if the person places down or leaves the object stationary on a surface in subsequent frames.\n"
        "   - Vehicle Rules: Predicates like 'get_on', 'get_off', 'ride', 'drive' MUST ONLY be used if the object is explicitly a vehicle (bicycle, car, motorcycle, bus, train) or an animal (horse).\n"
        "   - Ground Truth Fidelity: Strictly report visual facts. Do not hallucinate actions that are not visible.\n"
        "4. Output format MUST be strictly a valid JSON object matching this schema:\n"
        "{\n"
        '  "triplets": [\n'
        '    {\n'
        '      "subject": "[ID]",\n'
        '      "relation": "<predicate>",\n'
        '      "object": "[ID]",\n'
        '      "reason": "<brief explanation focusing strictly on the physical interaction between this subject and this object>"\n'
        '    }\n'
        '  ]\n'
        "}\n"
        "5. DO NOT output any markdown code blocks, explanations, or conversational text. Output ONLY the raw JSON object."
    )
    user_prompt = (
        "Analyze all provided sequential frames of this surveillance video clip.\n"
        "Examine active interactions between the marked entities [ID] across time.\n\n"
        "Examine active interactions between the marked entities across time.\n\n"
        "PREDEFINED RELATION TAXONOMY (CLOSED VOCABULARY):\n"
        "Every predicate in the 'relation' field MUST be an exact string match selected strictly from the 26 allowed categories: ['bite', 'carry', 'clean', 'cut', 'drive', 'feed', 'get_off', 'get_on', 'grab', 'hit', 'hold', 'hug', 'kick', 'kiss', 'knock', 'lean_on', 'lick', 'lift', 'play(instrument)', 'pull', 'push', 'ride', 'shake_hand_with', 'throw', 'touch', 'wave']. All out-of-vocabulary verbs are strictly prohibited.\n"
        "- Predicates like 'get_on', 'get_off', 'ride', 'drive' apply ONLY to vehicles or animals.\n"
        "- Categorize active physical contact between persons as 'touch'.\n"
        "- Categorize a person holding and transporting an object while moving as 'carry', and holding statically as 'hold'.\n"
        "- If an object remains stationary in the same location across all frames without movement, omit that pair.\n"
        "- If an active interaction occurs in any frames, report that relation even if it ends later.\n"
        "- In the 'reason' field, describe strictly the interaction between this subject and this object without referencing other entities.\n\n"
        'Respond strictly with the JSON object: {"triplets": [{"subject": "[ID]", "relation": "<verb>", "object": "[ID]", "reason": "..."}]}.'
    )

# 2. Nạp chính xác danh sách Frame ảnh (Chống nhiễm chéo frame giữa các video)
image_paths = []
if target_frame_names:
    for fn in target_frame_names:
        p = os.path.join('/content', fn)
        if os.path.exists(p):
            image_paths.append(p)
        else:
            matches = glob.glob(f'/content/**/{fn}', recursive=True)
            if matches:
                image_paths.append(matches[0])

if len(image_paths) == len(target_frame_names) and len(image_paths) > 0:
    print(f'✅ Đã khớp chính xác {len(image_paths)} frames visual prompt từ payload:')
else:
    all_imgs = sorted(glob.glob('/content/*.jpg') + glob.glob('/content/frames/*.jpg') + glob.glob('/content/frames/*/*.jpg'))
    image_paths = [p for p in all_imgs if 'vlm_' in os.path.basename(p) or 'frame' in os.path.basename(p)]
    if not image_paths:
        image_paths = all_imgs[:8]
    if not image_paths:
        raise FileNotFoundError('⚠️ Không tìm thấy ảnh .jpg nào trong /content/. Hãy tải các ảnh frame lên!')
    print(f'⚠️ Khớp fallback tìm thấy {len(image_paths)} frames ảnh:')

for p in image_paths:
    print(f'   - {os.path.basename(p)}')

# 3. Chuẩn bị nội dung tuần tự kèm mốc thời gian rõ ràng (Interleaved Temporal Anchoring)
user_content = []
for idx, p in enumerate(image_paths, 1):
    base_fn = os.path.basename(p)
    ts = base_fn.split("_")[-1].replace(".jpg", "") if "_" in base_fn and "s.jpg" in base_fn else f"{idx}s"
    user_content.append({"type": "text", "text": f"[Frame {idx} at timestamp {ts}]:"})
    user_content.append({"type": "image", "image": p})
user_content.append({"type": "text", "text": "\n" + user_prompt})

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_content}
]

# 4. Tiền xử lý dữ liệu và kích hoạt suy luận Video-LLM
text_prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
image_inputs, video_inputs = process_vision_info(messages)

proc_kwargs = {"text": [text_prompt], "padding": True, "return_tensors": "pt"}
if image_inputs is not None and len(image_inputs) > 0:
    proc_kwargs["images"] = image_inputs
if video_inputs is not None and len(video_inputs) > 0:
    proc_kwargs["videos"] = video_inputs

inputs = processor(**proc_kwargs).to("cuda")

# Đo lường Input Tokens (Prompt + 8 Frames)
input_tokens_count = inputs.input_ids.shape[1]

# Kích hoạt suy luận Pure Greedy Search (do_sample=False)
with torch.no_grad():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=False
    )

generated_ids_trimmed = [
    out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]

# Đo lường Output Tokens
output_tokens_count = len(generated_ids_trimmed[0])
total_tokens_count = input_tokens_count + output_tokens_count

response_text = processor.batch_decode(
    generated_ids_trimmed,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False
)[0]

# 5. Phân tích kết quả JSON và hiển thị trực quan
cleaned_json = response_text.strip()
if "```json" in cleaned_json:
    cleaned_json = cleaned_json.split("```json")[1].split("```")[0].strip()
elif "```" in cleaned_json:
    cleaned_json = cleaned_json.split("```")[1].split("```")[0].strip()

# 26 vi tu chuan cua du an (configs/relations.json)
valid_predicates = allowed_relations if allowed_relations else [
    'bite', 'carry', 'clean', 'cut', 'drive', 'feed', 'get_off', 'get_on', 'grab', 'hit',
    'hold', 'hug', 'kick', 'kiss', 'knock', 'lean_on', 'lick', 'lift', 'play(instrument)',
    'pull', 'push', 'ride', 'shake_hand_with', 'throw', 'touch', 'wave'
]

try:
    data = json.loads(cleaned_json)
    print("\n" + "="*80)
    print("KẾT QUẢ SUY LUẬN VLM (PREDICTED RELATION TRIPLETS + REASONING):")
    print("="*80)
    print(json.dumps(data.get("triplets", []), indent=2, ensure_ascii=False))

    triplets = data.get("triplets", [])
    print("-" * 80)
    print(f"{'Subject':<10} | {'Relation':<16} | {'Object':<10} | {'Status':<10} | Reason / Explanation")
    print("-" * 80)
    for trip in triplets:
        sub = trip.get("subject", "")
        rel = trip.get("relation", "")
        obj = trip.get("object", "")
        rsn = trip.get("reason", "")
        is_valid = rel in valid_predicates
        status = "[OK]" if is_valid else "[X] Invalid"
        print(f"{sub:<10} | {rel:<16} | {obj:<10} | {status:<10} | {rsn}")
    print("="*80)

    out_file = "/content/ket_qua_vlm.json"
    with open(out_file, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    print(f"✅ Đã lưu kết quả tự động vào: {out_file} ({len(triplets)} triplets)")

except Exception as e:
    print("⚠️ Phản hồi từ mô hình không đúng định dạng JSON chuẩn. Nội dung thô:")
    print(response_text)

# 6. Báo cáo định lượng tài nguyên Token (Benchmark Metrics)
print("="*80)
print("📊 THỐNG KÊ TÀI NGUYÊN TOKEN (BENCHMARK METRICS):")
print(f"   - Input Tokens (Prompt + 8 Frames):   {input_tokens_count:,} tokens")
print(f"   - Output Tokens (JSON Response):     {output_tokens_count:,} tokens")
print(f"   - Tổng số Token xử lý:               {total_tokens_count:,} tokens")
print("="*80)
